# DGL-COMP-01 / E00 — 合成VCF Baseline v0.2

**目的：** Colab上で、VCFの読み込み → 品質管理(QC) → 簡易候補順位付け → 公式12列CSV生成 → 形式チェックの**動作だけ**を確認します。

- **このノートブックには実患者データ、Hugging Face認証、ダウンロード処理は一切ありません。**
- ここで作成する `SYNTHETIC_NOT_FOR_SUBMISSION.csv` は公式提出不可です。**Submit & Scoreには絶対にアップロードしない**でください。
- `GENE_A` 等、HPO類似度、注釈、AFは架空です。順位やEPCRは臨床的根拠も校正も持ちません。
- 実データでの使用には、①利用規約の適合性確認 ②GRCh38参照配列との照合 ③妥当な機能注釈 ④HPOと疾患知識 ⑤遺伝形式・複合ヘテロ・位相 ⑥独立した品質検証が別途必要です。
- Google Driveのマウント、外部API、クラウドへの患者情報転送、公開リポジトリへのアップロードは行いません。

公式テンプレートの列：`proband_id,chrom_1,pos_1,ref_1,alt_1,chrom_2,pos_2,ref_2,alt_2,epcr,finding_type,notes`

## 1. ランタイムと出力先の確認

標準Pythonだけで動作します。外部ライブラリのインストールは必要ありません。

In [ ]:
import sys, csv, math, re
from pathlib import Path
from datetime import datetime, timezone

WORK = Path('/content/dgl_comp01_e00_synthetic') if Path('/content').exists() else Path('/tmp/dgl_comp01_e00_synthetic')
WORK.mkdir(parents=True, exist_ok=True)
print('E00 synthetic preflight')
print('Python:', sys.version.split()[0])
print('Output folder:', WORK)
print('Patient data access: DISABLED')

## 2. 完全な架空VCFを生成

公開リソースや患者データは取得しません。各レコードの `GENE`, `CSQ`, `AF` は、**アルゴリズム接続試験のために人工的に設定したINFOフィールド**です。実際のVCFに必ず存在するフィールドではありません。

In [ ]:
vcf_content = """##fileformat=VCFv4.2
##reference=GRCh38
##INFO=<ID=GENE,Number=1,Type=String,Description="Synthetic gene for pipeline testing">
##INFO=<ID=CSQ,Number=1,Type=String,Description="Synthetic variant consequence">
##INFO=<ID=AF,Number=1,Type=Float,Description="Synthetic population allele frequency">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype quality">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	SYNTHETIC_SAMPLE
chr1	110001	.	A	G	70	PASS	GENE=GENE_A;CSQ=missense_variant;AF=0.0004	GT:DP:GQ	0/1:30:99
chr2	120002	.	C	T	80	PASS	GENE=GENE_B;CSQ=stop_gained;AF=0.00002	GT:DP:GQ	0/1:25:95
chr3	130003	.	G	A	75	PASS	GENE=GENE_C;CSQ=synonymous_variant;AF=0.30	GT:DP:GQ	0/1:22:85
chr4	140004	.	T	C	20	PASS	GENE=GENE_D;CSQ=splice_donor_variant;AF=0.0001	GT:DP:GQ	0/1:13:50
chr5	150005	.	A	C	99	PASS	GENE=GENE_E;CSQ=frameshift_variant;AF=0.000004	GT:DP:GQ	1/1:40:95
chr6	160006	.	G	T	70	PASS	GENE=GENE_F;CSQ=missense_variant;AF=0.0007	GT:DP:GQ	0/1:18:45
chr7	170007	.	A	AT	90	PASS	GENE=GENE_G;CSQ=frameshift_variant;AF=0.0001	GT:DP:GQ	0/1:32:90
"""
vcf_path = WORK / 'SYNTHETIC_TEST_ONLY.vcf'
vcf_path.write_text(vcf_content, encoding='utf-8')
print('Synthetic VCF created (7 records).')

## 3. VCFを1レコードずつ読み込む

処理の単位を1行ずつにしておくと、将来、サイズの大きなVCFを扱う際も全行一括読込みを避けやすくなります。このデモでは単一サンプル・単一ALTだけを扱い、複数ALTや複数サンプルは解析対象外です。

In [ ]:
def parse_info(field):
    result = {}
    for token in field.split(';'):
        if '=' in token:
            k,v = token.split('=',1)
            result[k] = v
        elif token:
            result[token] = True
    return result

def iter_simple_vcf(vcf_path):
    sample_count = None
    with Path(vcf_path).open(encoding='utf-8') as f:
        for line in f:
            if line.startswith('##'):
                continue
            if line.startswith('#CHROM'):
                cols = line.rstrip('\n').split('\t')
                sample_count = len(cols)-9
                if sample_count != 1:
                    raise ValueError('Demo supports exactly one synthetic VCF sample')
                continue
            if not line.strip() or line.startswith('#'):
                continue
            cols = line.rstrip('\n').split('\t')
            if len(cols) != 10:
                raise ValueError('Expected exactly 10 VCF columns in demo')
            chrom, pos, vid, ref, alt, qual, flt, info_str, fmt, sample = cols
            if ',' in alt:
                raise ValueError('Multi-allelic variants not implemented in demo')
            keys, vals = fmt.split(':'), sample.split(':')
            sf = dict(zip(keys, vals))
            info = parse_info(info_str)
            yield dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                       qual=float(qual), filter=flt,
                       gene=info['GENE'], consequence=info['CSQ'], af=float(info['AF']),
                       gt=sf.get('GT','./.'), dp=int(sf.get('DP','0')),
                       gq=int(sf.get('GQ','0')))

records = list(iter_simple_vcf(vcf_path))
assert len(records) == 7
print('VCF parsing: PASS')
print('Synthetic record count:', len(records))

## 4. 品質管理（QC）と候補絞り込み

**この閾値は動作確認用の仮値です。** 実症例での適切な閾値はプラットフォーム・深度分布・遺伝形式・疾患機序から決めます。低QUALの変異や高頻度変異を、ここでは演習用に除外します。

In [ ]:
def qc_reason(r):
    if r['filter'] != 'PASS': return 'FILTER'
    if r['qual'] < 30: return 'QUAL'
    if r['dp'] < 10: return 'DP'
    if r['gq'] < 20: return 'GQ'
    if r['gt'] in ('./.', '.|.'): return 'missing_GT'
    if r['af'] > 0.01: return 'AF'
    return None

passed, excluded = [], []
for r in records:
    reason = qc_reason(r)
    if reason is None: passed.append(r)
    else: excluded.append(reason)
print('Input records:', len(records))
print('QC passed:', len(passed))
print('QC excluded:', len(excluded))
print('Exclusion reasons (synthetic):', {name:excluded.count(name) for name in sorted(set(excluded))})
assert len(passed) == 5 and len(excluded) == 2

## 5. 候補変異を順位付け

架空の遺伝子ごとに、人工的な「表現型関連度」（0〜1）を与え、予測機能影響・稀少性・遺伝子型と加重統合します。

**重要：** これはHPOの実装でも、ClinVar/gnomAD注釈でもありません。また、計算する `epcr` は公式提出項目に形を合わせた**デモ用順位スコア**です。実際の原因確率として解釈してはいけません。

In [ ]:
SYNTHETIC_PHENOTYPE_MATCH = {
    'GENE_A': .90, 'GENE_B': .80, 'GENE_C': .50,
    'GENE_D': .80, 'GENE_E': .55, 'GENE_F': .15, 'GENE_G': .70
}
SYNTHETIC_IMPACT = {
    'stop_gained':1.0,
    'frameshift_variant':1.0,
    'splice_donor_variant':.95,
    'missense_variant':.65,
    'synonymous_variant':.10,
}

def synthetic_score(r):
    phenotype = SYNTHETIC_PHENOTYPE_MATCH[r['gene']]
    impact = SYNTHETIC_IMPACT[r['consequence']]
    rarity = min(1.0, -math.log10(max(r['af'], 1e-6))/6.0)
    genotype = .8 if r['gt'] == '1/1' else .6
    return (.40*impact + .35*phenotype + .20*rarity + .05*genotype)

ranked = sorted(passed, key=lambda r: (-synthetic_score(r),r['chrom'],r['pos']))
assert len(ranked)==5
assert all(synthetic_score(ranked[i]) >= synthetic_score(ranked[i+1]) for i in range(len(ranked)-1))
print('Synthetic ranked candidates:', len(ranked))
print('Synthetic ranking pipeline: PASS')

## 6. 公式12列CSVを生成して形式検査

ファイル名とサンプルIDで、**合成データを誤って競技へ提出しない**よう明示します。

In [ ]:
COLUMNS = ['proband_id','chrom_1','pos_1','ref_1','alt_1',
           'chrom_2','pos_2','ref_2','alt_2','epcr','finding_type','notes']
output = WORK / 'SYNTHETIC_NOT_FOR_SUBMISSION.csv'
with output.open('w',newline='',encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    writer.writeheader()
    for r in ranked[:10]:
        writer.writerow({
          'proband_id':'SYNTHETIC_ONLY',
          'chrom_1':r['chrom'],'pos_1':r['pos'],
          'ref_1':r['ref'],'alt_1':r['alt'],
          'chrom_2':'','pos_2':'','ref_2':'','alt_2':'',
          'epcr':f'{synthetic_score(r):.6f}',
          'finding_type':'primary',
          'notes':'SYNTHETIC_TEST_DO_NOT_SUBMIT',
        })

def validate_demo_csv(path):
    with path.open(newline='',encoding='utf-8') as f:
        reader=csv.DictReader(f)
        assert reader.fieldnames == COLUMNS, 'Incorrect official column order'
        rows=list(reader)
    assert 1 <= len(rows) <= 10, 'Row count out of bounds'
    assert all(x['proband_id']=='SYNTHETIC_ONLY' for x in rows), 'Not a synthetic dataset'
    assert all(x['notes']=='SYNTHETIC_TEST_DO_NOT_SUBMIT' for x in rows), 'Safety marker missing'
    vals=[float(x['epcr']) for x in rows]
    assert all(math.isfinite(v) and 0<=v<=1 for v in vals)
    assert vals == sorted(vals,reverse=True)
    assert all(x['chrom_2']==x['pos_2']==x['ref_2']==x['alt_2']=='' for x in rows)
    return len(rows)

n = validate_demo_csv(output)
assert n == 5
print('Official header match: PASS (12 columns)')
print('Synthetic CSV validation: PASS')
print('Candidate rows:', n)
print('File:', output.name)
print('Official evaluator: NOT RUN')
print('Real-patient variant validation: NOT RUN')
print('DO NOT UPLOAD this synthetic CSV to Track 1.')

## 7. 実行結果の要約（患者情報を含まない）

次のセルの出力だけをChatGPTに共有できます。結果は**合成データの件数と処理状態だけ**です。

In [ ]:
summary = {
    'experiment': 'E00_SYNTHETIC_PREFLIGHT_v0.2',
    'input_records': len(records),
    'qc_passed': len(passed),
    'qc_excluded': len(excluded),
    'ranked_candidates': len(ranked),
    'official_columns': len(COLUMNS),
    'csv_validation': 'PASS' if n==5 else 'FAIL',
    'data': 'SYNTHETIC ONLY',
    'official_submission': 'NOT PERFORMED',
}
for k,v in summary.items():
    print(f'{k}: {v}')

## 8. 実データへの接続前に必要な研究課題

このノートブックの簡易注釈 `GENE/CSQ/AF` は人工的なもので、主催者の実VCFにあるとは限りません。**患者VCFをこのノートブックへ読み込ませないでください。**

本番E00に向け、独立した追加工程が必要です。

1. **VCF構造監査：** 参照ゲノム（GRCh38）、FORMAT、サンプル数、FILTER、注釈カラムの実在性を確認する。
2. **Annotation：** 公式データからの変異とClinVar、gnomAD、VEP等の正規の参照情報を、許可された環境・経路で結び付ける。
3. **HPOと遺伝形式：** 症状と疾患を対応付け、遺伝子順位・アレル状態・必要に応じてcompound-het位相を評価する。
4. **QCと再現性：** 深度・アレルバランス等の実際に存在する品質情報に合わせて基準を定め、false positivesを検査する。
5. **公式提出：** 根拠に基づく上位10件以内の候補変異、Methods Report、GitHub URLを準備し、ユーザー確認後に公式フォームへ提出する。

**患者データをColabへダウンロードする前に、主催者のDiscussion #30への回答と、Google側の適用条件を確認すること。** 合成データのみのこの実験を先に完了させる。